In [1]:
%load_ext sql
%config SqlMagic.autopandas = False
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# COVID-19 Data Pipeline — Week 10 Assignment

**Author:** Eslam Magdy
**instructor** Amr Ayyad
**Submission type:** Individual (solo)

## Objective
This notebook builds an end-to-end data pipeline for COVID-19 country-level statistics:
pulling live data from a public REST API, cleaning and transforming it with Pandas,
loading it into a PostgreSQL database, normalizing it into a Star Schema, and running
SQL-based analysis to extract insights.

## Data Source
- **API:** [COVID-19 Countries API](https://disease.sh/v3/covid-19/countries) (disease.sh)
- **Documentation:** [disease.sh API Docs](https://disease.sh/docs/#/COVID-19%3A%20Worldometers/get_v3_covid_19_countries)

## Pipeline Overview
1. **API & Data Collection** — Fetch live country data via `requests` and convert to a DataFrame
2. **Data Exploration** — Inspect structure, data types, and missing values
3. **Data Cleaning & Wrangling** — Remove invalid entries, fix data types, rename columns to be SQL-friendly
4. **PostgreSQL Loading** — Load the cleaned table into the `covid` schema via SQLAlchemy
5. **Data Normalization** — Split the flat table into a Star Schema (`dim_country`, `fact_covid_stats`) as views
6. **Data Analysis** — Run 6 analytical SQL queries using SQLMagic
7. **Findings / Insights** — Summarize key takeaways from the analysis

## Notes
- AI assistance was used in a limited capacity, restricted to the initial API GET request step, per assignment guidelines.

In [2]:
import pandas as pd
import os
import requests
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [3]:
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)
Magic= %sql $DATABASE_URL

## 1. API & Data Collection

In [4]:
url= "https://disease.sh/v3/covid-19/countries"
response = requests.get(url)
data = response.json()
print (data)


[{'updated': 1791319924978, 'country': 'Afghanistan', 'countryInfo': {'_id': 4, 'iso2': 'AF', 'iso3': 'AFG', 'lat': 33, 'long': 65, 'flag': 'https://disease.sh/assets/img/flags/af.png'}, 'cases': 234174, 'todayCases': 0, 'deaths': 7996, 'todayDeaths': 0, 'recovered': 211080, 'todayRecovered': 0, 'active': 15098, 'critical': 0, 'casesPerOneMillion': 5746, 'deathsPerOneMillion': 196, 'tests': 1390730, 'testsPerOneMillion': 34125, 'population': 40754388, 'continent': 'Asia', 'oneCasePerPeople': 174, 'oneDeathPerPeople': 5097, 'oneTestPerPeople': 29, 'activePerOneMillion': 370.46, 'recoveredPerOneMillion': 5179.32, 'criticalPerOneMillion': 0}, {'updated': 1791319924970, 'country': 'Albania', 'countryInfo': {'_id': 8, 'iso2': 'AL', 'iso3': 'ALB', 'lat': 41, 'long': 20, 'flag': 'https://disease.sh/assets/img/flags/al.png'}, 'cases': 334863, 'todayCases': 0, 'deaths': 3605, 'todayDeaths': 0, 'recovered': 330233, 'todayRecovered': 0, 'active': 1025, 'critical': 0, 'casesPerOneMillion': 116825,

In [5]:
# Convert the JSON data to a pandas DataFrame
db = pd.json_normalize(data)
db.head(5)

,updated,country,cases,todayCases,deaths,todayDeaths,recovered,todayRecovered,active,critical,...,oneTestPerPeople,activePerOneMillion,recoveredPerOneMillion,criticalPerOneMillion,countryInfo._id,countryInfo.iso2,countryInfo.iso3,countryInfo.lat,countryInfo.long,countryInfo.flag
0,1791319924978,Afghanistan,234174,0,7996,0,211080,0,15098,0,...,29,370.46,5179.32,0.0,4.0,AF,AFG,33.0,65.0,https://disease.sh/assets/img/flags/af.png
1,1791319924970,Albania,334863,0,3605,0,330233,0,1025,0,...,1,357.59,115209.32,0.0,8.0,AL,ALB,41.0,20.0,https://disease.sh/assets/img/flags/al.png
2,1791319924973,Algeria,272010,0,6881,0,183061,0,82068,0,...,196,1809.65,4036.61,0.0,12.0,DZ,DZA,28.0,3.0,https://disease.sh/assets/img/flags/dz.png
3,1791319925027,Andorra,48015,0,165,0,0,0,47850,0,...,0,617714.26,0.00,0.0,20.0,AD,AND,42.5,1.6,https://disease.sh/assets/img/flags/ad.png
4,1791319925006,Angola,107327,0,1937,0,103419,0,1971,0,...,23,56.27,2952.52,0.0,24.0,AO,AGO,-12.5,18.5,https://disease.sh/assets/img/flags/ao.png


## 2. Data Exploration

In [6]:
db.info()

<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   updated                 231 non-null    int64  
 1   country                 231 non-null    str    
 2   cases                   231 non-null    int64  
 3   todayCases              231 non-null    int64  
 4   deaths                  231 non-null    int64  
 5   todayDeaths             231 non-null    int64  
 6   recovered               231 non-null    int64  
 7   todayRecovered          231 non-null    int64  
 8   active                  231 non-null    int64  
 9   critical                231 non-null    int64  
 10  casesPerOneMillion      231 non-null    int64  
 11  deathsPerOneMillion     231 non-null    int64  
 12  tests                   231 non-null    int64  
 13  testsPerOneMillion      231 non-null    int64  
 14  population              231 non-null    int64  
 15  

In [7]:
# Check for missing values
db.isnull().sum()

updated                   0
country                   0
cases                     0
todayCases                0
deaths                    0
todayDeaths               0
recovered                 0
todayRecovered            0
active                    0
critical                  0
casesPerOneMillion        0
deathsPerOneMillion       0
tests                     0
testsPerOneMillion        0
population                0
continent                 0
oneCasePerPeople          0
oneDeathPerPeople         0
oneTestPerPeople          0
activePerOneMillion       0
recoveredPerOneMillion    0
criticalPerOneMillion     0
countryInfo._id           2
countryInfo.iso2          2
countryInfo.iso3          2
countryInfo.lat           0
countryInfo.long          0
countryInfo.flag          0
dtype: int64

In [8]:
db.columns.tolist()

['updated',
 'country',
 'cases',
 'todayCases',
 'deaths',
 'todayDeaths',
 'recovered',
 'todayRecovered',
 'active',
 'critical',
 'casesPerOneMillion',
 'deathsPerOneMillion',
 'tests',
 'testsPerOneMillion',
 'population',
 'continent',
 'oneCasePerPeople',
 'oneDeathPerPeople',
 'oneTestPerPeople',
 'activePerOneMillion',
 'recoveredPerOneMillion',
 'criticalPerOneMillion',
 'countryInfo._id',
 'countryInfo.iso2',
 'countryInfo.iso3',
 'countryInfo.lat',
 'countryInfo.long',
 'countryInfo.flag']

In [9]:
#check for missing values in the 'countryInfo.iso2' column
db[db['countryInfo.iso2'].isnull()][['country', 'continent', 'population']]

,country,continent,population
56,Diamond Princess,,0
119,MS Zaandam,,0


## 3. Data Cleaning & Wrangling

In [10]:
#change the type of the 'updated' column to datetime
db['updated'] = pd.to_datetime(db['updated'], unit='ms')

In [11]:
# Rename columns to snake_case
db = db.rename(columns={'countryInfo._id' : 'country_id',
                         'countryInfo.iso2': 'iso2',
                           'countryInfo.iso3': 'iso3',
                             'countryInfo.lat': 'lat', 
                             'countryInfo.long': 'long'})

In [12]:
# Drop the flag column
db = db.drop(columns=['countryInfo.flag'])

In [13]:
# Show all column names
db.columns.tolist()

['updated',
 'country',
 'cases',
 'todayCases',
 'deaths',
 'todayDeaths',
 'recovered',
 'todayRecovered',
 'active',
 'critical',
 'casesPerOneMillion',
 'deathsPerOneMillion',
 'tests',
 'testsPerOneMillion',
 'population',
 'continent',
 'oneCasePerPeople',
 'oneDeathPerPeople',
 'oneTestPerPeople',
 'activePerOneMillion',
 'recoveredPerOneMillion',
 'criticalPerOneMillion',
 'country_id',
 'iso2',
 'iso3',
 'lat',
 'long']

In [14]:
#Remove the rows for the cruise ships, These entries are not relevant for country-level analysis.
db = db[~db['country'].isin(['Diamond Princess', 'MS Zaandam'])].reset_index(drop=True)

In [15]:
db.shape
db.info()
db.dtypes

<class 'pandas.DataFrame'>
RangeIndex: 229 entries, 0 to 228
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   updated                 229 non-null    datetime64[ms]
 1   country                 229 non-null    str           
 2   cases                   229 non-null    int64         
 3   todayCases              229 non-null    int64         
 4   deaths                  229 non-null    int64         
 5   todayDeaths             229 non-null    int64         
 6   recovered               229 non-null    int64         
 7   todayRecovered          229 non-null    int64         
 8   active                  229 non-null    int64         
 9   critical                229 non-null    int64         
 10  casesPerOneMillion      229 non-null    int64         
 11  deathsPerOneMillion     229 non-null    int64         
 12  tests                   229 non-null    int64         
 13  t

updated                   datetime64[ms]
country                              str
cases                              int64
todayCases                         int64
deaths                             int64
todayDeaths                        int64
recovered                          int64
todayRecovered                     int64
active                             int64
critical                           int64
casesPerOneMillion                 int64
deathsPerOneMillion                int64
tests                              int64
testsPerOneMillion                 int64
population                         int64
continent                            str
oneCasePerPeople                   int64
oneDeathPerPeople                  int64
oneTestPerPeople                   int64
activePerOneMillion              float64
recoveredPerOneMillion           float64
criticalPerOneMillion            float64
country_id                       float64
iso2                                 str
iso3            

In [16]:
db.head(5)

,updated,country,cases,todayCases,deaths,todayDeaths,recovered,todayRecovered,active,critical,...,oneDeathPerPeople,oneTestPerPeople,activePerOneMillion,recoveredPerOneMillion,criticalPerOneMillion,country_id,iso2,iso3,lat,long
0,2026-10-06 20:52:04.978,Afghanistan,234174,0,7996,0,211080,0,15098,0,...,5097,29,370.46,5179.32,0.0,4.0,AF,AFG,33.0,65.0
1,2026-10-06 20:52:04.970,Albania,334863,0,3605,0,330233,0,1025,0,...,795,1,357.59,115209.32,0.0,8.0,AL,ALB,41.0,20.0
2,2026-10-06 20:52:04.973,Algeria,272010,0,6881,0,183061,0,82068,0,...,6591,196,1809.65,4036.61,0.0,12.0,DZ,DZA,28.0,3.0
3,2026-10-06 20:52:05.027,Andorra,48015,0,165,0,0,0,47850,0,...,469,0,617714.26,0.00,0.0,20.0,AD,AND,42.5,1.6
4,2026-10-06 20:52:05.006,Angola,107327,0,1937,0,103419,0,1971,0,...,18083,23,56.27,2952.52,0.0,24.0,AO,AGO,-12.5,18.5


In [17]:
# Show countries with zero population or zero tests
db[(db['population'] == 0) | (db['tests'] == 0)][['country', 'tests', 'population', 'cases']]

,country,tests,population,cases
44,Comoros,0,907419,9109
72,French Polynesia,0,284164,79254
88,Holy See (Vatican City State),0,799,29
106,Kiribati,0,123419,5085
126,Marshall Islands,0,60057,16138
141,N. Korea,0,25990679,4772813
148,Nicaragua,0,6779100,18491
151,Niue,0,1622,1059
170,Saint Helena,0,6115,2166
182,Seychelles,0,99426,51220


In [18]:
# Rename columns to snake_case
db = db.rename(columns={
    'todayCases': 'today_cases',
    'todayDeaths': 'today_deaths',
    'todayRecovered': 'today_recovered',
    'casesPerOneMillion': 'cases_per_million',
    'deathsPerOneMillion': 'deaths_per_million',
    'testsPerOneMillion': 'tests_per_million',
    'oneCasePerPeople': 'one_case_per_people',
    'oneDeathPerPeople': 'one_death_per_people',
    'oneTestPerPeople': 'one_test_per_people',
    'activePerOneMillion': 'active_per_million',
    'recoveredPerOneMillion': 'recovered_per_million',
    'criticalPerOneMillion': 'critical_per_million'
})

db.columns.tolist()

['updated',
 'country',
 'cases',
 'today_cases',
 'deaths',
 'today_deaths',
 'recovered',
 'today_recovered',
 'active',
 'critical',
 'cases_per_million',
 'deaths_per_million',
 'tests',
 'tests_per_million',
 'population',
 'continent',
 'one_case_per_people',
 'one_death_per_people',
 'one_test_per_people',
 'active_per_million',
 'recovered_per_million',
 'critical_per_million',
 'country_id',
 'iso2',
 'iso3',
 'lat',
 'long']

In [19]:
os.makedirs('outputs', exist_ok=True)
db.to_csv('outputs/cleaned_covid_data.csv', index=False)

## 4. PostgreSQL Loading

In [20]:
# Save the DataFrame to the database
db.to_sql(
    name='countries',
    schema='covid',
    con=engine,
    if_exists='replace',   # 'replace', 'append', or 'fail'
    index=False
)

engine.dispose()
print("Data saved to database")

InternalError: (psycopg2.errors.DependentObjectsStillExist) cannot drop table covid.countries because other objects depend on it
DETAIL:  view covid.dim_country depends on table covid.countries
view covid.fact_covid_stats depends on table covid.countries
view covid.summary_view depends on view covid.fact_covid_stats
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE covid.countries]
(Background on this error at: https://sqlalche.me/e/21/2j85)

## 5. Data Normalization

In [ ]:
%%sql
-- Create dimension table for countries
create VIEW covid.dim_country as 
select 
        country_id,
        country,
        continent,
        iso2,
        iso3,
        lat,
        long,
        population
from covid.countries

RuntimeError: (psycopg2.errors.DuplicateTable) relation "dim_country" already exists

[SQL: create VIEW covid.dim_country as
select
        country_id,
        country,
        continent,
        iso2,
        iso3,
        lat,
        long,
        population
from covid.countries]
(Background on this error at: https://sqlalche.me/e/21/f405)


In [ ]:
%%sql
-- Create fact table for COVID-19 statistics
CREATE VIEW covid.fact_covid_stats AS
SELECT
    country_id,
    updated,
    cases,
    today_cases,
    deaths,
    today_deaths,
    recovered,
    today_recovered,
    active,
    critical,
    tests,
    cases_per_million,
    deaths_per_million,
    tests_per_million,
    active_per_million,
    recovered_per_million,
    critical_per_million
FROM covid.countries;

RuntimeError: (psycopg2.errors.DuplicateTable) relation "fact_covid_stats" already exists

[SQL: CREATE VIEW covid.fact_covid_stats AS
SELECT
    country_id,
    updated,
    cases,
    today_cases,
    deaths,
    today_deaths,
    recovered,
    today_recovered,
    active,
    critical,
    tests,
    cases_per_million,
    deaths_per_million,
    tests_per_million,
    active_per_million,
    recovered_per_million,
    critical_per_million
FROM covid.countries;]
(Background on this error at: https://sqlalche.me/e/21/f405)


In [ ]:
%%sql
--check the data in the dimension and fact tables
select * from covid.dim_country;
select * from covid.fact_covid_stats;

-- check the connection between the dimension and fact tables
select d.country,f.cases,f.deaths
from covid.dim_country d
join covid.fact_covid_stats f on d.country_id = f.country_id
limit 10;

country,cases,deaths
Afghanistan,234174,7996
Albania,334863,3605
Algeria,272010,6881
Andorra,48015,165
Angola,107327,1937
Anguilla,3904,12
Antigua and Barbuda,9106,146
Argentina,10128845,130841
Armenia,451831,8777
Aruba,44224,292


## 6. Data Analysis

In [ ]:
%%sql
result1 << 
--Q1 — Rank Countries by Total Cases within Each Continent
SELECT d.country,d.continent,f.cases,
        rank() OVER(partition by d.continent order by f.cases desc) as case_rank_in_continent
    from covid.dim_country d
    join covid.fact_covid_stats f on d.country_id = f.country_id
    order by d.continent,case_rank_in_continent;

In [ ]:
result1.DataFrame().to_csv('outputs/rank_countries_by_total_cases.csv', index=False)
result1

country,continent,cases,case_rank_in_continent
South Africa,Africa,4076463,1
Morocco,Africa,1278992,2
Tunisia,Africa,1153361,3
Egypt,Africa,516023,4
Libyan Arab Jamahiriya,Africa,507274,5
Ethiopia,Africa,501157,6
Réunion,Africa,494595,7
Zambia,Africa,349304,8
Kenya,Africa,344130,9
Botswana,Africa,330638,10


In [ ]:
%%sql
result2 << 
--Q2— Case Fatality Rate (CFR) per Country
SELECT d.country, 
            round(f.deaths::numeric / nullif(f.cases, 0) * 100, 2) AS case_fatality_rate
    FROM covid.fact_covid_stats f
    join covid.dim_country d on d.country_id = f.country_id
order by case_fatality_rate desc
limit 10;

In [ ]:
result2.DataFrame().to_csv('outputs/Case Fatality Rate (CFR) per Country.csv', index=False)
result2

country,case_fatality_rate
Yemen,18.07
Western Sahara,10.00
Sudan,7.89
Syrian Arab Republic,5.48
Somalia,4.98
Peru,4.86
Egypt,4.77
Mexico,4.35
Bosnia,4.06
Liberia,3.65


In [ ]:
%%sql
result3 <<
--Q3— Relationship Between Population and Total Cases
select d.country, d.continent, d.population, f.cases,
        round(f.cases::NUMERIC / NULLIF(d.population,0) * 100 ,2) as cases_pct_of_population,
        f.cases_per_million
FROM covid.dim_country d
join covid.fact_covid_stats f on d.country_id = f.country_id
WHERE d.population > 0
ORDER BY cases_pct_of_population DESC
LIMIT 10;

In [ ]:
result3.DataFrame().to_csv('outputs/Relationship Between Population and Total Cases.csv', index=False)
result3

country,continent,population,cases,cases_pct_of_population,cases_per_million
Brunei,Asia,445431,343719,77.17,771655
San Marino,Europe,34085,26185,76.82,768226
Faroe Islands,Europe,49233,34658,70.40,703959
S. Korea,Asia,51329899,34571873,67.35,673523
Austria,Europe,9066710,6081287,67.07,670727
Niue,Australia-Oceania,1622,1059,65.29,652898
Slovenia,Europe,2078034,1356546,65.28,652803
Andorra,Europe,77463,48015,61.98,619844
Martinique,North America,374087,230354,61.58,615777
France,Europe,65584518,40138560,61.20,612013


In [ ]:
%%sql
result4 <<
--Q4— Recovery Rate per Country
select d.country,d.continent,f.recovered,f.cases,
            round(f.recovered::NUMERIC / NULLIF(f.cases,0) *100 ,2) as recovery_rate
from covid.fact_covid_stats f
join covid.dim_country d on d.country_id = f.country_id
order by recovery_rate desc
limit 10;


In [ ]:
result4.DataFrame().to_csv('outputs/Recovery Rate per Country.csv', index=False)
result4

country,continent,recovered,cases,recovery_rate
Holy See (Vatican City State),Europe,29,29,100.00
N. Korea,Asia,4772739,4772813,100.00
Falkland Islands (Malvinas),South America,1930,1930,100.00
Singapore,Asia,3004131,3006155,99.93
S. Korea,Asia,34535939,34571873,99.90
Marshall Islands,Australia-Oceania,16121,16138,99.89
Qatar,Asia,513687,514524,99.84
Taiwan,Asia,10222518,10241523,99.81
Cyprus,Asia,679745,681110,99.80
Palau,Australia-Oceania,6276,6290,99.78


In [ ]:
%%sql
result5 <<
--Q5—Continent-Level Summary (Aggregation)
select d.continent,
        sum(d.population) as total_population,
        count(d.country) as total_countries,
        sum(f.cases) as total_cases,
        sum(f.deaths) as total_deaths,
        sum(f.recovered) as total_recovered,
        round(avg(f.cases_per_million), 2) as avg_cases_per_million,
        round(avg(f.deaths_per_million), 2) as avg_deaths_per_million,
        round(sum(f.deaths)::NUMERIC / nullif(sum(f.cases),0) * 100 ,2) as case_fatality_rate,
        round(sum(f.recovered)::NUMERIC / nullif(sum(f.cases),0) * 100 ,2) as recovery_rate
from covid.fact_covid_stats f
join covid.dim_country d on d.country_id = f.country_id
group by d.continent
order by total_cases desc; 

In [ ]:
result5.DataFrame().to_csv('outputs/Continent-Level Summary.csv', index=False)
result5

continent,total_population,total_countries,total_cases,total_deaths,total_recovered,avg_cases_per_million,avg_deaths_per_million,case_fatality_rate,recovery_rate
Europe,747543837,48,253406198,2101824,235797174,399994.79,2697.88,0.83,93.05
Asia,4711356783,50,221500265,1553662,108524029,164630.02,704.44,0.70,48.99
North America,598140916,39,131889132,1695941,126483239,228463.90,1536.85,1.29,95.90
South America,437694443,14,70200879,1367332,65935495,187173.21,2372.43,1.95,93.92
Australia-Oceania,43470408,20,14895771,33015,14615509,250161.40,485.00,0.22,98.12
Africa,1406728744,58,12860924,258892,9211514,43286.41,319.91,2.01,71.62


In [ ]:
%%sql
result6 <<
--Q6—Testing Coverage vs. Cases Detected
SELECT
    d.country,
    d.continent,
    d.population,
    f.tests,
    f.cases,
    round(f.tests::NUMERIC / NULLIF(d.population,0) * 100 ,2) as tests_pct_of_population,
    round(f.cases::NUMERIC / NULLIF(d.population,0) * 100 ,2) as cases_pct_of_population,
    round(f.cases::numeric / NULLIF(f.tests, 0) * 100, 2) AS positivity_rate_pct
FROM covid.dim_country d
join covid.fact_covid_stats f on d.country_id = f.country_id
WHERE f.tests > 0
order by positivity_rate_pct desc
limit 10;

In [ ]:
result6.DataFrame().to_csv('outputs/Testing Coverage vs. Cases Detected.csv', index=False)
result6

country,continent,population,tests,cases,tests_pct_of_population,cases_pct_of_population,positivity_rate_pct
Macao,Asia,667490,7850,27673,1.18,4.15,352.52
S. Korea,Asia,51329899,15804065,34571873,30.79,67.35,218.75
Algeria,Africa,45350148,230960,272010,0.51,0.60,117.77
New Caledonia,Australia-Oceania,290915,98964,80064,34.02,27.52,80.90
Brazil,South America,215353593,63776166,38743918,29.61,17.99,60.75
Antigua and Barbuda,North America,99509,18901,9106,18.99,9.15,48.18
Vanuatu,Australia-Oceania,321832,24976,12019,7.76,3.73,48.12
Brunei,Asia,445431,717784,343719,161.14,77.17,47.89
Slovenia,Europe,2078034,2847701,1356546,137.04,65.28,47.64
Bolivia,South America,11992656,2710261,1212131,22.60,10.11,44.72


In [ ]:
%%sql
--Summary View
CREATE VIEW covid.summary_view AS
SELECT country,
       continent,
       population,
       cases,
       deaths,
       recovered,
       tests,
       f.active,
       round(cases::NUMERIC / NULLIF(population,0) * 100 ,2) as cases_pct_of_population,
       round(deaths::NUMERIC / NULLIF(population,0) * 100 ,2) as deaths_pct_of_population,
       round(recovered::NUMERIC / NULLIF(population,0) * 100 ,2) as recovered_pct_of_population,
       round(tests::NUMERIC / NULLIF(population,0) * 100 ,2) as tests_pct_of_population,
       round(deaths::numeric / nullif(cases, 0) * 100, 2) AS case_fatality_rate,
       round(recovered::numeric / nullif(cases, 0) * 100, 2) AS recovery_rate
FROM covid.dim_country d
JOIN covid.fact_covid_stats f ON d.country_id = f.country_id;


RuntimeError: (Your query contains named parameters (NUMERIC, NUMERIC, NUMERIC, NUMERIC, numeric, numeric) but the named parameters feature is "warn". 
Enable it with: %config SqlMagic.named_parameters="enabled" 
or disable it with: %config SqlMagic.named_parameters="disabled"
For more info, see the docs: https://jupysql.ploomber.io/en/latest/api/configuration.html#named-parameters)
(psycopg2.errors.DuplicateTable) relation "summary_view" already exists

[SQL: CREATE VIEW covid.summary_view AS
SELECT country,
       continent,
       population,
       cases,
       deaths,
       recovered,
       tests,
       f.active,
       round(cases::NUMERIC / NULLIF(population,0) * 100 ,2) as cases_pct_of_population,
       round(deaths::NUMERIC / NULLIF(population,0) * 100 ,2) as deaths_pct_of_population,
       round(recovered::NUMERIC / NULLIF(population,0) * 100 ,2) as recovered_pct_of_population,
       round(tests::NUMERIC / NULLIF(population,0) * 100 ,2) as tests_pct_of_population,
    

In [ ]:
%%sql
result7 <<
--to check the summary view
select * from covid.summary_view ORDER BY cases desc limit 10;

In [ ]:
result7.DataFrame().to_csv('outputs/summary view.csv', index=False)
result7

country,continent,population,cases,deaths,recovered,tests,active,cases_pct_of_population,deaths_pct_of_population,recovered_pct_of_population,tests_pct_of_population,case_fatality_rate,recovery_rate
USA,North America,334805269,111820082,1219487,109814428,1186851502,786167,33.40,0.36,32.80,354.49,1.09,98.21
India,Asia,1406631776,45035393,533570,0,935879495,44501823,3.20,0.04,0.00,66.53,1.18,0.00
France,Europe,65584518,40138560,167642,39970918,271490188,0,61.20,0.26,60.95,413.95,0.42,99.58
Germany,Europe,83883596,38828995,183027,38240600,122332384,405368,46.29,0.22,45.59,145.84,0.47,98.48
Brazil,South America,215353593,38743918,711380,36249161,63776166,1783377,17.99,0.33,16.83,29.61,1.84,93.56
S. Korea,Asia,51329899,34571873,35934,34535939,15804065,0,67.35,0.07,67.28,30.79,0.10,99.90
Japan,Asia,125584838,33803572,74694,0,100414883,33728878,26.92,0.06,0.00,79.96,0.22,0.00
Italy,Europe,60262770,26723249,196487,26361218,281126449,165544,44.34,0.33,43.74,466.50,0.74,98.65
UK,Europe,68497907,24910387,232112,24678275,522526476,0,36.37,0.34,36.03,762.84,0.93,99.07
Russia,Europe,145805947,24124215,402756,23545818,273400000,175641,16.55,0.28,16.15,187.51,1.67,97.60


## Findings / Insights

**1. Highest Cases by Continent (Query 1)**
South Africa recorded the highest total cases within the African continent (4,076,463 cases), ranking #1 among African countries.

**2. Case Fatality Rate (Query 2)**
Yemen showed an unexpectedly high case fatality rate relative to its reported case count. This is likely not a sign that the virus was more lethal there, but rather a data-quality effect: due to a weak healthcare system and ongoing conflict, many mild or moderate cases were never tested or recorded, so only severe/critical cases made it into the official count — inflating the deaths-to-cases ratio.

**3. Population vs. Case Percentage (Query 3)**
There is no direct relationship between a country's population *size* and its case percentage of population. The countries with the highest `cases_pct_of_population` are mostly very small nations (e.g., Niue with only 1,622 people, Faroe Islands, San Marino) — a handful of cases there translates into a large percentage. However, there are notable exceptions: South Korea (~51M) and Austria (~9M) also appear high on this list, showing that large populations can still experience high infection percentages.

**4. Recovery Rate (Query 4)**
The highest recovery rates (100%) appeared in Holy See (Vatican City, Europe), N. Korea (Asia), and Falkland Islands (South America). However, a 100% recovery rate across multiple unrelated countries is a sign of a data reporting pattern rather than a genuine public health outcome — it likely means these countries stopped updating `recovered` independently from `cases` at some point during the pandemic, so the system simply set recovered = cases.

**5. Continent-Level Summary (Query 5)**
Europe had the highest totals overall, with 253,406,198 total cases and 2,101,824